In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [2]:
import sys
sys.path.append('../')
from model.lrp import LRP
from model.mmst_lrp import MMST_ViT_LRP

import Inference as inf
from src import dataloader
from model import configs, engine

In [3]:
import os
seed = 1987 
import torch  
import numpy as np
torch.manual_seed(seed)
np.random.seed(seed)
from IPython.core.display import HTML, display
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import tiktoken  
from typing import Union, Dict, List
import pandas as pd
from PyPDF2 import PdfReader

### Text

In [4]:
LOW_EXTREME_RANGE_BLOCKS = [32016, 32017, 32018, 32019, 382016, 1032016, 1072016]
COMMON_RANGE_BLOCKS = [72016, 72019, 122016, 122017, 122018, 122019, 142016, 142017, 142018, 142019, 162016, 162017, 162018, 172016, 172017, 172018, 182016, 182017, 182018, 182019, 382017, 382018, 682016, 682017, 762016, 
                       762017, 762018, 762019, 1022016, 1022017, 1022018, 1022019, 1032017, 1032018, 1032019, 1072017, 1072018, 1072019, 1112016, 1112017, 1112018, 1112019, 1762017, 1762018, 1762019]
HIGH_EXTREME_RANGE_BLOCKS = [72017, 72018, 162019]

class TextScoresAnalysis():
    def __init__(self, exp_name: str, batch_size: int, layer: int, head:int, lrp: str = False):
        self.exp_name = exp_name
        self.batch_size = batch_size
        self.head = head
        self.layer = layer

        self.root_dir = '/data2/hkaman/Projects/ViT/EXPs/July'
        self.attn_dir = os.path.join(self.root_dir , 'EXP_' + self.exp_name + '/attn_scores')

        self.df = pd.read_csv(os.path.join(self.root_dir, 'EXP_' + self.exp_name, self.exp_name + '_train.csv'))
        self.TextEncoder = tiktoken.get_encoding('gpt2')

    def extreme_categ_visualize_text(self):

        main_dict = self._all_blcok_scores()

        low_extreme_dict = {key: main_dict[key] for key in LOW_EXTREME_RANGE_BLOCKS if key in main_dict}
        common_dict = {key: main_dict[key] for key in COMMON_RANGE_BLOCKS if key in main_dict}
        high_extreme_dict = {key: main_dict[key] for key in HIGH_EXTREME_RANGE_BLOCKS if key in main_dict}

        low_mean = self._calculate_mean_for_dict(low_extreme_dict)[-1, :, :, self.layer]
        low_mean = low_mean[2:, 2:]
        low_mean = np.mean(low_mean, axis=0)
        low_mean = self._normalize_array(low_mean)

        common_mean = self._calculate_mean_for_dict(common_dict)[-1, :, :, self.layer]
        common_mean = common_mean[2:, 2:]
        common_mean = np.mean(common_mean, axis=0)
        common_mean = self._normalize_array(common_mean)

        high_mean = self._calculate_mean_for_dict(high_extreme_dict)[-1, :, :, self.layer]
        high_mean = high_mean[2:, 2:]
        high_mean = np.mean(high_mean, axis=0)
        high_mean = self._normalize_array(high_mean)

        words = self._text_decode(block = 162017)
        self._visualize_text(words, low_mean)
        self._visualize_text(words, common_mean)
        self._visualize_text(words, high_mean)

    def single_visualize_text(self, block: None):
        """Use decoded text and attention scores to create a visualization."""
        words = self._text_decode(block = block)
        importances = self._single_calc_attn(block = block)
        self._visualize_text(words, importances)
    
    def _calculate_mean_for_dict(self, dictionary):

        all_values = []
        for key, values in dictionary.items():
            all_values.append(values)
        
        # Assuming all values are numeric matrices of the same shape
        if all_values:
            mean_value = np.mean(all_values, axis=0)
        else:
            mean_value = None  # or handle empty case as needed
        
        return mean_value
   
    def _find_file_in_directory(self, block):
        """
        Search for a specific .txt file in a given directory and return its full path.
        
        :param directory: Directory to search in
        :param filename: Name of the file to search for
        :return: Full path of the file if found, else None
        """

        directory = '/data2/hkaman/Data/Livingston/text'
        # Ensure the filename ends with .txt
        block = str(block)
        if not block.endswith('.txt'):
            block_root_name = block[:-4]

            if len(str(block_root_name)) == 1:
                block_fullnames = 'LIV_00' + str(block_root_name) + '.txt'
            elif len(str(block_root_name)) == 2:
                block_fullnames = 'LIV_0' + str(block_root_name) + '.txt'
            elif len(str(block_root_name)) == 3:
                block_fullnames = 'LIV_' + str(block_root_name) + '.txt'

        
        # Search for the file in the directory
        for root, _, files in os.walk(directory):
            if block_fullnames in files:
                return os.path.join(root, block_fullnames)
        
        return None

    def _return_text_of_block(self, block):
        # Extract the file extension to determine how to process it
        # block_root_name = block_name[:-4] + 
        text_path = self._find_file_in_directory(block = block)

        _, file_extension = os.path.splitext(text_path)
        
        if file_extension.lower() == '.pdf':
            # Handle PDF files
            pdf_loader = PdfReader(open(text_path, "rb"))
            file_text = ""
            for page_num in range(len(pdf_loader.pages)):
                pdf_page = pdf_loader.pages[page_num]
                if pdf_page.extract_text() is not None:
                    file_text += pdf_page.extract_text()
            return file_text
        
        elif file_extension.lower() == '.txt':
            # Handle text files
            with open(text_path, "r", encoding="utf-8") as file:
                file_text = file.read()
            return file_text
        
        else:
            # Unsupported file type
            raise ValueError("Unsupported file format: " + file_extension)

    def _text_encode(self, block):

        # text = self._return_text_of_block(block = block)
        # encoded_texts = [self.TextEncoder.encode(text) for text in text]
        # max_length = 249 
        # padded_texts = [text[:max_length] + [0] * (max_length - len(text)) for text in encoded_texts]
        # print(padded_texts)
        # tokens = np.array(padded_texts, dtype = np.uint32)#.to(self.device)
        arr = np.load('/home/hkaman/Documents/multimodel-transformers-vye/Junky/attn_scores.npy', allow_pickle=True).item()
        tokens = arr[10]['tokens']
        return tokens

    def _text_decode(self, block):
        """Decode tokens using the TextEncoder."""
        # Decode both the text and get token offsets
        tokens = self._text_encode(block = block)
        # print(len(tokens), tokens)
        # decoded_text = self.TextEncoder.decode(tokens)
        token_bytes = [self.TextEncoder.decode_single_token_bytes(token) for token in tokens]
        words = [t.decode('utf-8') for t in token_bytes]
        return words
    
    def _single_calc_attn(self, block):
        """Calculate the average attention score for a specific layer and head, excluding the first token."""
        # Extract the attention scores for the specified head and layer
        # if ana_status == 'single': 
        attn_scores = self._single_blcok_scores(block = block)[self.head, :, :, self.layer]#self.attns_arr[self.head, :, :, self.layer]
        attn_scores = attn_scores[2:, 2:]
        avg_attn_scores = np.mean(attn_scores, axis=0)
        norm_attn_scores = self._normalize_array(avg_attn_scores)
        return norm_attn_scores
        
    def _global_cal_attn(self):
        # elif ana_status == 'global':
        scores = self._all_blcok_scores()
        return scores
    
    def _normalize_array(self, values):
        min_old = values.min()
        max_old = values.max()
        min_new, max_new = -1, 1
        normalized_values = [(value - min_old) / (max_old - min_old) * (max_new - min_new) + min_new for value in values]
        return np.array(normalized_values, dtype= np.float32)
    
    def _format_special_tokens(self, word):
        # Strip underscores often used in tokenized outputs
        return word.replace('_', ' ')

    def _get_color(self, attr):
        # clip values to prevent CSS errors (Values should be from [-1,1])
        attr = max(-1, min(1, attr))
        if attr > 0:
            hue = 120
            sat = 75
            lig = 100 - int(50 * attr)
        else:
            hue = 0
            sat = 75
            lig = 100 - int(-40 * attr)
        return "hsl({}, {}%, {}%)".format(hue, sat, lig)

    def _format_word_importances(self, words, importances):
        tags = ["<td>"]
        for word, importance in zip(words, importances):
            color = self._get_color(importance)
            tags.append(
                '<mark style="background-color: {color}; opacity:1.0; line-height:1.75">'
                '<font color="black"> {word} </font></mark>'.format(color=color, word=word)
            )
        tags.append("</td>")
        return "".join(tags)

    def _visualize_text(self, words, importances, legend=True):
        assert len(words) == len(importances), "Words and importances must have the same length."
        
        dom = ["<table style='width: 100%;'>"]
        dom.append(
            "<tr>{}</tr>".format(self._format_word_importances(words, importances))
        )
        
        if legend:
            dom.append(
                '<div style="border-top: 1px solid; margin-top: 5px; padding-top: 5px; display: inline-block">'
            )
            dom.append("<b>Legend: </b>")
            for value, label in zip([-1, 0, 1], ["Negative", "Neutral", "Positive"]):
                dom.append(
                    '<span style="display: inline-block; width: 20px; height: 10px; border: 1px solid; background-color: {value};"></span> {label}  '.format(
                        value=self._get_color(value), label=label
                    )
                )
            dom.append("</div>")
        
        dom.append("</table>")
        html = HTML("".join(dom))
        display(html)
        return html
    
    def _return_blocks_patch_info(self):
        unique_blocks = pd.unique(self.df['block'])
        block_patch_range = {}

        lower_ = 0
        for idx, block in enumerate(unique_blocks):
            size = len(self.df[self.df['block'] == block]) / 256
            block_patch_range[block] = (lower_, size + lower_)
            lower_ += size

        return block_patch_range
    
    def _single_blcok_scores(self, block):


        range_dict = self._return_blocks_patch_info()

        key = block
        lower_bound = int(range_dict[key][0])
        upper_bound = int(range_dict[key][1])

        # Calculate the starting and ending file indices
        start_file_index = lower_bound // self.batch_size
        end_file_index = (upper_bound - 1) // self.batch_size  # Subtract 1 to handle inclusive upper bound correctly

        # List to store slices of arrays for averaging
        slices = []

        # Loop over the necessary file indices
        for file_index in range(start_file_index, end_file_index + 1):
            filename = f"train_attn_scores_{file_index}.npy"
            file_path = os.path.join(self.attn_dir, filename)
            
            if os.path.exists(file_path):
                array = np.load(file_path)

                # Calculate slice bounds within the current array
                slice_start = lower_bound - self.batch_size * file_index
                slice_end = upper_bound - self.batch_size * file_index

                # Adjust slice bounds to fit within the current array
                slice_start = max(0, slice_start)
                slice_end = min(self.batch_size, slice_end)

                if slice_start < slice_end:  # Ensure there is something to slice
                    slices.append(array[slice_start:slice_end])

        # Concatenate all slices along the first axis and compute the mean
        if slices:
            combined_array = np.concatenate(slices, axis=0)
            mean_array = np.percentile(combined_array, 95, axis=0)#np.mean(combined_array, axis=0)

        return mean_array
        
    def _all_blcok_scores(self):
        scores = {}

        range_dict = self._return_blocks_patch_info()
        # Iterate over each specified range
        for key, bounds in range_dict.items():
            lower_bound = int(bounds[0])
            upper_bound = int(bounds[1])

            # Calculate the starting and ending file indices
            start_file_index = lower_bound // self.batch_size
            end_file_index = (upper_bound - 1) // self.batch_size  # Subtract 1 to handle inclusive upper bound correctly

            # List to store slices of arrays for averaging
            slices = []

            # Loop over the necessary file indices
            for file_index in range(start_file_index, end_file_index + 1):
                filename = f"train_attn_scores_{file_index}.npy"
                file_path = os.path.join(self.attn_dir, filename)
                
                if os.path.exists(file_path):
                    array = np.load(file_path)

                    # Calculate slice bounds within the current array
                    slice_start = lower_bound - self.batch_size * file_index
                    slice_end = upper_bound - self.batch_size * file_index

                    # Adjust slice bounds to fit within the current array
                    slice_start = max(0, slice_start)
                    slice_end = min(self.batch_size, slice_end)

                    if slice_start < slice_end:  # Ensure there is something to slice
                        slices.append(array[slice_start:slice_end])

            # Concatenate all slices along the first axis and compute the mean
            if slices:
                combined_array = np.concatenate(slices, axis=0)
                mean_array = np.mean(combined_array, axis=0) #np.percentile(combined_array, 95, axis=0)
                scores[key] = mean_array

        return scores
    

In [5]:
exp_name = '005_S2S1MT_0001_01_8_6_768_30_64_MSE_Q_Norm_Init_context128'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 162018)

In [6]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 32017)

#### GPT2: 000_S2S1MT_0001_01_8_6_768_30_48_MSE

In [10]:
exp_name = '000_S2S1MT_0001_01_8_6_768_30_48_MSE'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).single_visualize_text(block = 162018)

In [11]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).single_visualize_text(block = 32017)

In [12]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).single_visualize_text(block = 72018)

In [10]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 48, layer = -1, head = -1, lrp = False).extreme_categ_visualize_text()

#### BERT

In [6]:
exp_name = '002_S2S1MT_0001_01_8_6_768_30_32_MSE_BERT'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 162018)

In [7]:
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).single_visualize_text(block = 32017)

In [14]:
exp_name = '002_S2S1MT_0001_01_8_6_768_30_32_MSE_BERT'
_ = TextScoresAnalysis(exp_name= exp_name, batch_size = 64, layer = -1, head = -1, lrp = False).extreme_categ_visualize_text()

### Others

In [50]:
class TextAttenVis(nn.Module):
    def __init__(self, tokens: List[int], attns_arr, layer: int, head: int):
        super().__init__()
        self.tokens = tokens
        self.attns_arr = attns_arr  # Assuming attns_arr is a tensor with shape [H, X, Y, L]
        self.layer = layer
        self.head = head
        self.TextEncoder = tiktoken.get_encoding('gpt2')
        
    def decode(self):
        """Decode tokens using the TextEncoder."""
        # Decode both the text and get token offsets
        decoded_text = self.TextEncoder.decode(self.tokens)
        token_bytes = [self.TextEncoder.decode_single_token_bytes(token) for token in self.tokens]
        words = [t.decode('utf-8') for t in token_bytes]
        return words
    
    def calc_attn(self):
        """Calculate the average attention score for a specific layer and head, excluding the first token."""
        # Extract the attention scores for the specified head and layer
        attn_scores = self.attns_arr[self.head, :, :, self.layer]
        # Exclude the first token's attention scores (position 0)
        attn_scores = attn_scores[2:, 2:]
        # Calculate the average across columns
        avg_attn_scores = np.mean(attn_scores, axis=0)

        norm_attn_scores = self.normalize_array(avg_attn_scores)
        return norm_attn_scores
    
    def plot(self):
        """Use decoded text and attention scores to create a visualization."""
        words = self.decode()
        importances = self.calc_attn()

        self.visualize_text(words, importances)

    def normalize_array(self, values):
        min_old = values.min()
        max_old = values.max()
        min_new, max_new =-1, 1
        normalized_values = [(value - min_old) / (max_old - min_old) * (max_new - min_new) + min_new for value in values]
        return np.array(normalized_values, dtype= np.float32)
    
    def format_special_tokens(self, word):
        # Strip underscores often used in tokenized outputs
        return word.replace('_', ' ')


    def _get_color(self, attr):
        # clip values to prevent CSS errors (Values should be from [-1,1])
        attr = max(-1, min(1, attr))
        if attr > 0:
            hue = 120
            sat = 75
            lig = 100 - int(50 * attr)
        else:
            hue = 0
            sat = 75
            lig = 100 - int(-40 * attr)
        return "hsl({}, {}%, {}%)".format(hue, sat, lig)

    def format_word_importances(self, words, importances):
        tags = ["<td>"]
        for word, importance in zip(words, importances):
            color = self._get_color(importance)
            tags.append(
                '<mark style="background-color: {color}; opacity:1.0; line-height:1.75">'
                '<font color="black"> {word} </font></mark>'.format(color=color, word=word)
            )
        tags.append("</td>")
        return "".join(tags)

    def visualize_text(self, words, importances, legend=True):
        assert len(words) == len(importances), "Words and importances must have the same length."
        
        dom = ["<table style='width: 100%;'>"]
        dom.append(
            "<tr>{}</tr>".format(self.format_word_importances(words, importances))
        )
        
        if legend:
            dom.append(
                '<div style="border-top: 1px solid; margin-top: 5px; padding-top: 5px; display: inline-block">'
            )
            dom.append("<b>Legend: </b>")
            for value, label in zip([-1, 0, 1], ["Negative", "Neutral", "Positive"]):
                dom.append(
                    '<span style="display: inline-block; width: 20px; height: 10px; border: 1px solid; background-color: {value};"></span> {label}  '.format(
                        value=self._get_color(value), label=label
                    )
                )
            dom.append("</div>")
        
        dom.append("</table>")
        html = HTML("".join(dom))
        display(html)
        return html

In [5]:
import pandas as pd
exp_df = pd.read_csv('/data2/hkaman/Projects/ViT/EXPs/July/EXP_00_all_lr0001_wd01_dr30_768_8_6_64_yz/00_all_lr0001_wd01_dr30_768_8_6_64_yz_train.csv')
exp_df

,Unnamed: 0,block,cultivar,x,y,ytrue,ypred_w1,ypred_w2,ypred_w3,ypred_w4,...,ypred_w6,ypred_w7,ypred_w8,ypred_w9,ypred_w10,ypred_w11,ypred_w12,ypred_w13,ypred_w14,ypred_w15
0,0,1762018,3,20,0,12.932465,12.296338,12.329173,12.563106,12.336212,...,12.361913,12.381466,12.320471,12.399814,12.465119,12.320650,12.465689,12.417679,12.434526,12.338538
1,1,1762018,3,20,1,13.512397,12.336802,12.398656,12.621766,12.381386,...,12.431398,12.410377,12.365887,12.423085,12.526346,12.348647,12.536717,12.483647,12.498887,12.374754
2,2,1762018,3,20,2,13.879617,12.411592,12.465762,12.706409,12.471702,...,12.473975,12.493038,12.423740,12.504116,12.577646,12.421958,12.589268,12.549308,12.572040,12.465219
3,3,1762018,3,20,3,14.297155,12.407248,12.454989,12.677439,12.428853,...,12.470098,12.469646,12.437248,12.512933,12.576036,12.428190,12.582082,12.554774,12.565578,12.468667
4,4,1762018,3,20,4,13.703508,12.458765,12.508912,12.761071,12.526074,...,12.536178,12.549388,12.501978,12.554325,12.649865,12.485950,12.647227,12.616670,12.639295,12.516236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3551995,3551995,182017,3,53,25,9.634854,8.471873,8.608835,8.475002,8.566308,...,8.481845,8.625408,8.540093,8.385203,8.752691,8.593314,8.728557,8.637101,8.517480,8.426453
3551996,3551996,182017,3,53,26,9.700796,8.493179,8.671465,8.502865,8.585302,...,8.487825,8.668302,8.543048,8.420088,8.773476,8.590909,8.758548,8.669851,8.572538,8.437601
3551997,3551997,182017,3,53,27,9.907112,8.485063,8.674589,8.517700,8.529644,...,8.474715,8.680202,8.541413,8.352735,8.746447,8.555909,8.744676,8.696676,8.535196,8.414667
3551998,3551998,182017,3,53,28,10.553209,8.446444,8.639217,8.507057,8.565543,...,8.457965,8.646433,8.561075,8.397346,8.706067,8.540813,8.706911,8.657640,8.533182,8.444560


In [6]:
unique_blocks = pd.unique(exp_df['block'])
block_sizes = {}

lower_ = 0
for idx, block in enumerate(unique_blocks):
    size = len(exp_df[exp_df['block'] == block]) / 256
    block_sizes[block] = (lower_, size + lower_)
    lower_ += size

print(block_sizes)

{1762018: (0, 10.0), 1762019: (10.0, 20.0), 162017: (20.0, 74.0), 1022019: (74.0, 300.0), 1032018: (300.0, 817.0), 1112016: (817.0, 858.0), 162019: (858.0, 920.0), 182016: (920.0, 1579.0), 1032019: (1579.0, 2097.0), 762017: (2097.0, 2118.0), 32017: (2118.0, 2503.0), 72016: (2503.0, 2855.0), 32019: (2855.0, 3248.0), 1022017: (3248.0, 3353.0), 162018: (3353.0, 3411.0), 762019: (3411.0, 3900.0), 172016: (3900.0, 3922.0), 122019: (3922.0, 4402.0), 72018: (4402.0, 4735.0), 1072018: (4735.0, 5068.0), 182018: (5068.0, 5727.0), 142017: (5727.0, 5888.0), 382016: (5888.0, 6214.0), 182019: (6214.0, 6893.0), 1032016: (6893.0, 7373.0), 72017: (7373.0, 7598.0), 122016: (7598.0, 8079.0), 682016: (8079.0, 8103.0), 762018: (8103.0, 8591.0), 682017: (8591.0, 8614.0), 142019: (8614.0, 8787.0), 1072017: (8787.0, 9102.0), 1022018: (9102.0, 9311.0), 122018: (9311.0, 9791.0), 142016: (9791.0, 9956.0), 72019: (9956.0, 10291.0), 172018: (10291.0, 10313.0), 1022016: (10313.0, 10536.0), 762016: (10536.0, 11030.0

In [7]:
def process_arrays(directory, range_dict):
    results = {}

    # Iterate over each specified range
    for key, bounds in range_dict.items():
        lower_bound = int(bounds[0])
        upper_bound = int(bounds[1])
        print(lower_bound, upper_bound)
        # Calculate the starting and ending file indices
        start_file_index = lower_bound // 64
        end_file_index = (upper_bound - 1) // 64  # Subtract 1 to handle inclusive upper bound correctly

        # List to store slices of arrays for averaging
        slices = []

        # Loop over the necessary file indices
        for file_index in range(start_file_index, end_file_index + 1):
            filename = f"train_attn_scores_{file_index}.npy"
            file_path = os.path.join(directory, filename)
            
            if os.path.exists(file_path):
                array = np.load(file_path)

                # Calculate slice bounds within the current array
                slice_start = lower_bound - 64 * file_index
                slice_end = upper_bound - 64 * file_index

                # Adjust slice bounds to fit within the current array
                slice_start = max(0, slice_start)
                slice_end = min(64, slice_end)

                if slice_start < slice_end:  # Ensure there is something to slice
                    slices.append(array[slice_start:slice_end])

        # Concatenate all slices along the first axis and compute the mean
        if slices:
            combined_array = np.concatenate(slices, axis=0)
            mean_array = np.percentile(combined_array, 95, axis=0)#np.mean(combined_array, axis=0)
            results[key] = mean_array

    return results


In [8]:
root_dir = '/data2/hkaman/Projects/ViT/EXPs/July/attnscores'

results = process_arrays(root_dir, block_sizes)
for range_key, mean_array in results.items():
    print(f"Mean for {range_key}: {mean_array.shape}")

0 10
10 20
20 74
74 300
300 817
817 858
858 920
920 1579
1579 2097
2097 2118
2118 2503
2503 2855
2855 3248
3248 3353
3353 3411
3411 3900
3900 3922
3922 4402
4402 4735
4735 5068
5068 5727
5727 5888
5888 6214
6214 6893
6893 7373
7373 7598
7598 8079
8079 8103
8103 8591
8591 8614
8614 8787
8787 9102
9102 9311
9311 9791
9791 9956
9956 10291
10291 10313
10313 10536
10536 11030
11030 11376
11376 11416
11416 11589
11589 11630
11630 11640
11640 12050
12050 12371
12371 12408
12408 12430
12430 12489
12489 12880
12880 12894
12894 13133
13133 13433
13433 13589
13589 13875
Mean for 1762018: (8, 250, 250, 6)
Mean for 1762019: (8, 250, 250, 6)
Mean for 162017: (8, 250, 250, 6)
Mean for 1022019: (8, 250, 250, 6)
Mean for 1032018: (8, 250, 250, 6)
Mean for 1112016: (8, 250, 250, 6)
Mean for 162019: (8, 250, 250, 6)
Mean for 182016: (8, 250, 250, 6)
Mean for 1032019: (8, 250, 250, 6)
Mean for 762017: (8, 250, 250, 6)
Mean for 32017: (8, 250, 250, 6)
Mean for 72016: (8, 250, 250, 6)
Mean for 32019: (8, 25

In [9]:
results[162018]

array([[[[3.12077114e-04, 2.05452670e-03, 4.05650351e-03,
          4.11857723e-03, 3.93479352e-03, 3.99169698e-03],
         [2.87346095e-02, 2.85659153e-02, 1.16173336e-02,
          7.65043031e-03, 4.90558473e-03, 4.12883610e-03],
         [2.95295771e-02, 2.98675355e-02, 1.19761853e-02,
          7.81004969e-03, 4.94389143e-03, 4.13521379e-03],
         ...,
         [6.94386545e-04, 6.80358557e-04, 5.76887606e-03,
          6.08793832e-03, 5.09310467e-03, 4.14722459e-03],
         [6.94386545e-04, 6.80358557e-04, 5.76887606e-03,
          6.08793832e-03, 5.09310467e-03, 4.14722459e-03],
         [6.94386545e-04, 6.80358557e-04, 5.76887606e-03,
          6.08793832e-03, 5.09310467e-03, 4.14722459e-03]],

        [[4.09277707e-01, 1.02099190e-02, 4.16409830e-03,
          4.47429297e-03, 4.26150160e-03, 4.04873258e-03],
         [4.21366596e-04, 2.72258658e-05, 2.11745594e-03,
          2.18103826e-03, 3.95511556e-03, 3.99382738e-03],
         [4.04213060e-04, 2.60279830e-05, 2.0363

In [44]:
arr = np.load('/home/hkaman/Documents/multimodel-transformers-vye/Junky/attn_scores.npy', allow_pickle=True).item()
arr[10]['tokens']

In [46]:
len(arr[10]['tokens'])

248

In [41]:
len(arr[17]['tokens']), results[382017].shape

(245, (8, 250, 250, 6))

In [48]:
_ = TextAttenVis(arr[17]['tokens'], results[1032019], layer = -1, head= -1).plot()

In [51]:
_ = TextAttenVis(arr[10]['tokens'], results[162018], layer = -1, head= -1).plot()

In [23]:
_ = TextAttenVis(text_attns[10]['tokens'], text_attns[10]['text_array'], layer = -1, head= -1).plot()

### LRP

In [4]:
config = configs.Configs(
    img_size = 16, 
    patch_size = 8, 
    embed_dim = 768, 
    mlp_dim = 512, 
    pool = 'cls',
    in_channels = 8,
    out_channels = 1, 
    num_heads = 8, 
    num_layers = 6, 
    cond = False,
    multi_conv = False,
    attn_dropout = 0.3, 
    proj_dropout = 0.3, 
    drop_path = 0.0,
    post_norm = False, 
    vis = True, 
    tokenizer = 'EC',
    mask_modality = None
    ).call()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MMST_ViT_LRP(config, cond = False).to(device)

exp = '08_MMST_LRP_Test'
exp_output_dir = '/data2/hkaman/Projects/ViT/EXPs/' + 'EXP_' + exp
best_model_name = os.path.join(exp_output_dir, 'best_model_' + exp + '.pth')

model.load_state_dict(torch.load(best_model_name))

<All keys matched successfully>

In [6]:
model.eval()
attribution_generator = LRP(model)

In [7]:
data_loader_training, data_loader_validate, data_loader_test = dataloader.dataloaders(
    batch_size = 1, 
    img_size = 16,
    in_channels = 8, 
    resmapling_status = False,
    data = 's2',
    exp_name = 'test'
    )

(13875, 36) | (8233, 36) | (12443, 36)


In [8]:
data_dict_stt = {}
for batch, sample in enumerate(data_loader_training):
    data_dict_stt[sample['block'][0]] = {
    'image': sample['image'][0].to(device),
    'met':sample['met'][0].to(device),
    'text': sample['EmbText'],
    'yz': sample['YZ'][0],
    'mask':sample['mask'][0]
}

In [ ]:
for block, data in data_dict_stt.items():
    # print(data['image'].unsqueeze(0).shape, data['met'].unsqueeze(0).shape, data['yz'].unsqueeze(0).shape)
    imgmet_attr, context_attr = attribution_generator.generate_LRP(data['image'].unsqueeze(0), 
                                                                 data['text'], 
                                                                 data['met'].unsqueeze(0), 
                                                                 data['yz'].unsqueeze(0),)
    
    print(imgmet_attr[0].shape, imgmet_attr[1].shape, context_attr.shape)
    # transformer_attribution = transformer_attribution.reshape(1, 4, 4, 4)
    # transformer_attribution4 = (transformer_attribution - transformer_attribution.min()) / (transformer_attribution.max() - transformer_attribution.min())
    # data_dict_stt[date_time]['lpr'] = transformer_attribution4.data.cpu().numpy()